In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F

In [ ]:
# 1. Inicializar la Sesión de Spark
# Asegúrate de tener PySpark instalado y configurado.
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()


behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)
clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)

#clientes_psdf = clientes_df.pandas_api()
#behavioural_psdf = behavioural_df.pandas_api()

/opt/conda/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


### ANALISIS BEHAVIOURAL


In [6]:
## 📊 ANALISIS BEHAVIOURAL - Exploración Inicial

# Reemplaza behavioural_psdf.head() con la función nativa show()
print("Primeras 5 filas de BEHAVIOURAL:")
behavioural_df.show(5, truncate=False)

# Además, añadimos printSchema() para revisar tipos (fundamental en Big Data)
print("Esquema de BEHAVIOURAL:")
behavioural_df.printSchema()

Primeras 5 filas de BEHAVIOURAL:
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821016961u00XXX|ES182147947X|2020-08-22|0.0                 |2700.0           |0.0                     |0.0                 |0.0                     |0.0            

In [7]:
## 🔍 Estadísticas Descriptivas (PySpark nativo)

# Contar filas (reemplaza shape[0])
n_filas_beh = behavioural_df.count()

print(f"Shape: ({n_filas_beh}, {len(behavioural_df.columns)})")
print("\nEstadísticas Descriptivas:")
# Utilizamos el describe() nativo de PySpark, más eficiente para grandes datasets
behavioural_df.describe().show(truncate=False)

# También puedes mostrar los tipos usando printSchema() si quieres una lista:
print("\nTipos de Datos:")
behavioural_df.printSchema()

Shape: (1724854, 14)

Estadísticas Descriptivas:
+-------+------------------+------------+--------------------+------------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+------------------+------------------+--------+
|summary|CONTRACT_ID       |CLIENT_ID   |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT |CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS   |NUMBER_INSTALMENTS|CURRENCY|
+-------+------------------+------------+--------------------+------------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+------------------+------------------+--------+
|count  |1724854           |1724854     |1724854             |1724854           |1724854                 |1724854             |1724854                

In [11]:
## 🗑️ Recuento de Valores Nulos (Optimizado para PySpark)

# Crea una lista de expresiones de agregación: 
# (Si la columna es nula, cuenta 1, si no, cuenta 0)
nulos_expr = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in behavioural_df.columns
]

# Ejecuta la agregación una sola vez en el cluster
missing_values_beh = behavioural_df.agg(*nulos_expr).collect()[0]

print("Valores nulos por columna en BEHAVIOURAL:")
found_missing = False
for column in behavioural_df.columns:
    missing_count = missing_values_beh[column]
    if missing_count > 0:
        porcentaje = (missing_count / n_filas_beh) * 100
        print(f"  {column}: {missing_count} nulos ({porcentaje:.2f}%)")
        found_missing = True

if not found_missing:
    print("No hay valores nulos en el dataset BEHAVIOURAL.")

Valores nulos por columna en BEHAVIOURAL:
No hay valores nulos en el dataset BEHAVIOURAL.


### ANALISIS CLIENTS

In [12]:
## 👤 ANALISIS CLIENTS - Exploración Inicial

# Reemplaza clientes_psdf.head()
print("Primeras 5 filas de CLIENTS:")
clientes_df.show(5, truncate=False)

# Mostrar el esquema y tipos de datos
print("Esquema de CLIENTS:")
clientes_df.printSchema()

Primeras 5 filas de CLIENTS:
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_S

In [13]:
## 🔍 Estadísticas Descriptivas de CLIENTS

n_filas_cli = clientes_df.count()

print(f"Shape: ({n_filas_cli}, {len(clientes_df.columns)})")
print("\nEstadísticas Descriptivas:")
clientes_df.describe().show(truncate=False)

Shape: (162977, 45)

Estadísticas Descriptivas:
+-------+------------+----------------------+-----------------+------+------------------+------------------+------------------+---------------------+--------------+-----------------------+--------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+---------------------+-------------------+---------------------+------------------+---------------+-------------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+------------------+-------------------+------------------+
|summary|CLIENT_ID   |NON_

In [15]:
## 🗑️ Recuento de Valores Nulos en CLIENTS (Optimizado para PySpark)

# Crea una lista de expresiones de agregación para CLIENTS
nulos_expr_cli = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in clientes_df.columns
]

# Ejecuta la agregación
missing_values_cli = clientes_df.agg(*nulos_expr_cli).collect()[0]

print("Valores nulos por columna en CLIENTS:")
found_missing_cli = False
for column in clientes_df.columns:
    missing_count = missing_values_cli[column]
    if missing_count > 0:
        porcentaje = (missing_count / n_filas_cli) * 100
        print(f"  {column}: {missing_count} nulos ({porcentaje:.2f}%)")
        found_missing_cli = True

if not found_missing_cli:
    print("No hay valores nulos en el dataset CLIENTS.")

Valores nulos por columna en CLIENTS:
  INSTALLMENT: 7 nulos (0.00%)
  EDUCATION: 39640 nulos (24.32%)
  MARITAL_STATUS: 2 nulos (0.00%)
  JOB_SENIORITY: 29174 nulos (17.90%)
  CAR_AGE: 107550 nulos (65.99%)
  FAMILY_SIZE: 2 nulos (0.00%)
  REACTIVE_SCORING: 91901 nulos (56.39%)
  PROACTIVE_SCORING: 337 nulos (0.21%)
  BEHAVIORAL_SCORING: 32246 nulos (19.79%)
  DAYS_LAST_INFO_CHANGE: 1 nulos (0.00%)
  NUMBER_OF_PRODUCTS: 21903 nulos (13.44%)
  EMPLOYER_ORGANIZATION_TYPE: 29464 nulos (18.08%)
  NUM_PREVIOUS_LOAN_APP: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_MAX: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_MIN: 8770 nulos (5.38%)
  LOAN_ANNUITY_PAYMENT_SUM: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_MAX: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_MIN: 8770 nulos (5.38%)
  LOAN_APPLICATION_AMOUNT_SUM: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_MAX: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_MIN: 8770 nulos (5.38%)
  LOAN_CREDIT_GRANTED_SUM: 8770 nulos (5.38%)
  LOAN_VARIABLE_RATE_MAX: 8770

### DATASETS DE ID'S QUE COINCIDEN EN AMBOS DATASETS

In [17]:
## 🔗 Coincidencias de CLIENT_ID (Uso de Joins, como en CP5)

# 1. Contar cuántos clientes de CLIENTS tienen registros en BEHAVIOURAL (Inner Join)
# Esto indica que el cliente es activo en el dataset de comportamiento
df_coincidencias = clientes_df.alias("c").join(
    behavioural_df.alias("b"),
    on="CLIENT_ID",
    how="inner"
).select(F.col("c.CLIENT_ID")).distinct()

n_coincidencias = df_coincidencias.count()
n_clientes_totales = clientes_df.count()

print(f"Clientes en CLIENTS con datos en BEHAVIOURAL (Inner Join): {n_coincidencias}")
print(f"Clientes Únicos en CLIENTS: {n_clientes_totales}")

# 2. Calcular Clientes 'desaparecidos' o sin comportamiento:
# Usamos LEFT ANTI JOIN para encontrar IDs en CLIENTS que NO están en BEHAVIOURAL.
# Esto es más preciso que la resta de conteos brutos.
clientes_sin_comportamiento = clientes_df.join(
    behavioural_df,
    on="CLIENT_ID",
    how="left_anti"
).count()

print(f"\nClientes en CLIENTS sin registros de BEHAVIOURAL (Left Anti Join): {clientes_sin_comportamiento}")


Clientes en CLIENTS con datos en BEHAVIOURAL (Inner Join): 46046
Clientes Únicos en CLIENTS: 162977

Clientes en CLIENTS sin registros de BEHAVIOURAL (Left Anti Join): 116931
